In [1]:
!kaggle competitions download -c playground-series-s5e4
!unzip *.zip

 98%|█████████████████████████████████████▍| 22.0M/22.3M [00:03<00:00, 8.06MB/s]
100%|██████████████████████████████████████| 22.3M/22.3M [00:03<00:00, 7.02MB/s]
Archive:  playground-series-s5e4.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [ ]:
from pathlib import Path
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import missingno

from sklearn import set_config
set_config(transform_output = "pandas")

from sklearn.model_selection import ShuffleSplit, KFold, StratifiedKFold
from sklearn.model_selection import cross_validate, GridSearchCV

from sklearn.feature_selection import SelectFromModel, RFECV

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import LinearSVC
from lightgbm import LGBMClassifier
from xgboost import XGBRegressor
from catboost import CatBoostClassifier


KAGGLE_RUN = False
if KAGGLE_RUN:
    working_dir = Path('/kaggle/input/playground-series-s5e4')
else:
    working_dir = Path().cwd()


In [3]:
!head train.csv

id,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes
0,Mystery Matters,Episode 98,,True Crime,74.81,Thursday,Night,,0.0,Positive,31.41998
1,Joke Junction,Episode 26,119.8,Comedy,66.95,Saturday,Afternoon,75.95,2.0,Negative,88.01241
2,Study Sessions,Episode 16,73.9,Education,69.97,Tuesday,Evening,8.97,0.0,Negative,44.92531
3,Digital Digest,Episode 45,67.17,Technology,57.22,Monday,Morning,78.7,2.0,Positive,46.27824
4,Mind & Body,Episode 86,110.51,Health,80.07,Monday,Afternoon,58.68,3.0,Neutral,75.61031
5,Fitness First,Episode 19,26.54,Health,48.96,Saturday,Afternoon,,3.0,Positive,22.77047
6,Criminal Minds,Episode 47,69.83,True Crime,35.82,Sunday,Night,39.02,0.0,Neutral,64.75024
7,News Roundup,Episode 44,48.52,News,44.99,Thursday,Night,20.12,0.0,Positive,22.37517
8,Daily Digest,Episode 32,105.87,News,69.81,Monday,Evening,,2.0,Neutral,68.00124


In [4]:

train_df = pd.read_csv(working_dir/'train.csv', index_col='id')
train_df

,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes
id,,,,,,,,,,,
0,Mystery Matters,Episode 98,NaN,True Crime,74.81,Thursday,Night,NaN,0.0,Positive,31.41998
1,Joke Junction,Episode 26,119.80,Comedy,66.95,Saturday,Afternoon,75.95,2.0,Negative,88.01241
2,Study Sessions,Episode 16,73.90,Education,69.97,Tuesday,Evening,8.97,0.0,Negative,44.92531
3,Digital Digest,Episode 45,67.17,Technology,57.22,Monday,Morning,78.70,2.0,Positive,46.27824
4,Mind & Body,Episode 86,110.51,Health,80.07,Monday,Afternoon,58.68,3.0,Neutral,75.61031
...,...,...,...,...,...,...,...,...,...,...,...
749995,Learning Lab,Episode 25,75.66,Education,69.36,Saturday,Morning,NaN,0.0,Negative,56.87058
749996,Business Briefs,Episode 21,75.75,Business,35.21,Saturday,Night,NaN,2.0,Neutral,45.46242
749997,Lifestyle Lounge,Episode 51,30.98,Lifestyle,78.58,Thursday,Morning,84.89,0.0,Negative,15.26000


In [5]:
test_df = pd.read_csv(working_dir/'test.csv', index_col='id')
test_df

,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment
id,,,,,,,,,,
750000,Educational Nuggets,Episode 73,78.96,Education,38.11,Saturday,Evening,53.33,1.0,Neutral
750001,Sound Waves,Episode 23,27.87,Music,71.29,Sunday,Morning,NaN,0.0,Neutral
750002,Joke Junction,Episode 11,69.10,Comedy,67.89,Friday,Evening,97.51,0.0,Positive
750003,Comedy Corner,Episode 73,115.39,Comedy,23.40,Sunday,Morning,51.75,2.0,Positive
750004,Life Lessons,Episode 50,72.32,Lifestyle,58.10,Wednesday,Morning,11.30,2.0,Neutral
...,...,...,...,...,...,...,...,...,...,...
999995,Mind & Body,Episode 100,21.05,Health,65.77,Saturday,Evening,96.40,3.0,Negative
999996,Joke Junction,Episode 85,85.50,Comedy,41.47,Saturday,Night,30.52,2.0,Negative
999997,Joke Junction,Episode 63,12.11,Comedy,25.92,Thursday,Evening,73.69,1.0,Neutral


In [6]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 750000 entries, 0 to 749999
Data columns (total 11 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Podcast_Name                 750000 non-null  object 
 1   Episode_Title                750000 non-null  object 
 2   Episode_Length_minutes       662907 non-null  float64
 3   Genre                        750000 non-null  object 
 4   Host_Popularity_percentage   750000 non-null  float64
 5   Publication_Day              750000 non-null  object 
 6   Publication_Time             750000 non-null  object 
 7   Guest_Popularity_percentage  603970 non-null  float64
 8   Number_of_Ads                749999 non-null  float64
 9   Episode_Sentiment            750000 non-null  object 
 10  Listening_Time_minutes       750000 non-null  float64
dtypes: float64(5), object(6)
memory usage: 68.7+ MB


In [7]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 250000 entries, 750000 to 999999
Data columns (total 10 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Podcast_Name                 250000 non-null  object 
 1   Episode_Title                250000 non-null  object 
 2   Episode_Length_minutes       221264 non-null  float64
 3   Genre                        250000 non-null  object 
 4   Host_Popularity_percentage   250000 non-null  float64
 5   Publication_Day              250000 non-null  object 
 6   Publication_Time             250000 non-null  object 
 7   Guest_Popularity_percentage  201168 non-null  float64
 8   Number_of_Ads                250000 non-null  float64
 9   Episode_Sentiment            250000 non-null  object 
dtypes: float64(4), object(6)
memory usage: 21.0+ MB


In [9]:

train_df.describe()


,Episode_Length_minutes,Host_Popularity_percentage,Guest_Popularity_percentage,Number_of_Ads,Listening_Time_minutes
count,662907.000000,750000.000000,603970.000000,749999.000000,750000.000000
mean,64.504738,59.859901,52.236449,1.348855,45.437406
std,32.969603,22.873098,28.451241,1.151130,27.138306
min,0.000000,1.300000,0.000000,0.000000,0.000000
25%,35.730000,39.410000,28.380000,0.000000,23.178350
50%,63.840000,60.050000,53.580000,1.000000,43.379460
75%,94.070000,79.530000,76.600000,2.000000,64.811580
max,325.240000,119.460000,119.910000,103.910000,119.970000


In [10]:
test_df.describe()

,Episode_Length_minutes,Host_Popularity_percentage,Guest_Popularity_percentage,Number_of_Ads
count,2.212640e+05,250000.000000,201168.000000,250000.000000
mean,4.192987e+02,59.716491,52.192796,1.355852
std,1.668545e+05,22.880028,28.445034,4.274399
min,2.470000e+00,2.490000,0.000000,0.000000
25%,3.578000e+01,39.250000,28.320000,0.000000
50%,6.397000e+01,59.900000,53.360000,1.000000
75%,9.415000e+01,79.390000,76.560000,2.000000
max,7.848626e+07,117.760000,116.820000,2063.000000


In [ ]:
NUMERIC_COLUMNS=['day', 'pressure', 'maxtemp', 'temparature', 'mintemp', 'dewpoint', 'humidity', 'cloud', 'sunshine', 'winddirection', 'windspeed']
CATEGORIC_COLUMNS=[]
TARGET_COLUMN=['']
ALL_COLUMNS=NUMERIC_COLUMNS+CATEGORIC_COLUMNS+TARGET_COLUMN

In [ ]:
target = train_df[TARGET_COLUMN]
train = train_df.drop(columns=TARGET_COLUMN)
test = test_df


In [21]:
train

,country,store,product,day,week,weekday,month,year,day_sin,day_cos,month_sin,month_cos,year_sin,year_cos,holiday
id,,,,,,,,,,,,,,,
1,Canada,Discount Stickers,Kaggle,1,53,4,1,2010,0.017213,0.999852,5.000000e-01,0.866025,7.818315e-01,0.62349,True
2,Canada,Discount Stickers,Kaggle Tiers,1,53,4,1,2010,0.017213,0.999852,5.000000e-01,0.866025,7.818315e-01,0.62349,True
3,Canada,Discount Stickers,Kerneler,1,53,4,1,2010,0.017213,0.999852,5.000000e-01,0.866025,7.818315e-01,0.62349,True
4,Canada,Discount Stickers,Kerneler Dark Mode,1,53,4,1,2010,0.017213,0.999852,5.000000e-01,0.866025,7.818315e-01,0.62349,True
5,Canada,Stickers for Less,Holographic Goose,1,53,4,1,2010,0.017213,0.999852,5.000000e-01,0.866025,7.818315e-01,0.62349,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230125,Singapore,Premium Sticker Mart,Holographic Goose,31,52,5,12,2016,0.508671,0.860961,-2.449294e-16,1.000000,-7.053966e-14,1.00000,False
230126,Singapore,Premium Sticker Mart,Kaggle,31,52,5,12,2016,0.508671,0.860961,-2.449294e-16,1.000000,-7.053966e-14,1.00000,False
230127,Singapore,Premium Sticker Mart,Kaggle Tiers,31,52,5,12,2016,0.508671,0.860961,-2.449294e-16,1.000000,-7.053966e-14,1.00000,False


In [22]:
target

id
1          973.0
2          906.0
3          423.0
4          491.0
5          300.0
           ...  
230125     466.0
230126    2907.0
230127    2299.0
230128    1242.0
230129    1622.0
Name: num_sold, Length: 221259, dtype: float64

In [23]:
test

,country,store,product,day,week,weekday,month,year,day_sin,day_cos,month_sin,month_cos,year_sin,year_cos,holiday
id,,,,,,,,,,,,,,,
230130,Canada,Discount Stickers,Holographic Goose,1,52,6,1,2017,0.017213,0.999852,5.000000e-01,0.866025,0.781831,0.623490,True
230131,Canada,Discount Stickers,Kaggle,1,52,6,1,2017,0.017213,0.999852,5.000000e-01,0.866025,0.781831,0.623490,True
230132,Canada,Discount Stickers,Kaggle Tiers,1,52,6,1,2017,0.017213,0.999852,5.000000e-01,0.866025,0.781831,0.623490,True
230133,Canada,Discount Stickers,Kerneler,1,52,6,1,2017,0.017213,0.999852,5.000000e-01,0.866025,0.781831,0.623490,True
230134,Canada,Discount Stickers,Kerneler Dark Mode,1,52,6,1,2017,0.017213,0.999852,5.000000e-01,0.866025,0.781831,0.623490,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
328675,Singapore,Premium Sticker Mart,Holographic Goose,31,1,1,12,2019,0.508671,0.860961,-2.449294e-16,1.000000,0.433884,-0.900969,False
328676,Singapore,Premium Sticker Mart,Kaggle,31,1,1,12,2019,0.508671,0.860961,-2.449294e-16,1.000000,0.433884,-0.900969,False
328677,Singapore,Premium Sticker Mart,Kaggle Tiers,31,1,1,12,2019,0.508671,0.860961,-2.449294e-16,1.000000,0.433884,-0.900969,False


In [ ]:
transformer = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), NUMERIC_COLUMNS),
        ('categories', OneHotEncoder(sparse_output=False), CATEGORIC_COLUMNS),
    ], remainder='passthrough'
)

classifier = XGBRegressor()

pipe = Pipeline(
    steps=[
        ('transform_columns', transformer),
        ('classifier', classifier)
        ]
        )


In [ ]:
cv_results = cross_validate(
    pipe,
    train,
    target,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=13),
    scoring="roc_auc",
    n_jobs=2
)

errors_tree_regressor = pd.Series(
    cv_results["test_score"]
)
errors_tree_regressor.describe()

count    3.000000
mean     0.050040
std      0.000187
min      0.049919
25%      0.049932
50%      0.049945
75%      0.050100
max      0.050255
Name: Decision tree regressor, dtype: float64

In [ ]:
sub_df = pd.DataFrame(
    index=test.index,
    data={
        TARGET_COLUMN[0]:pipe.predict(test)
    },
)
sub_df    


In [30]:
if KAGGLE_RUN:
    sub_df.to_csv("/kaggle/working/submission.csv")
    !head /kaggle/working/submission.csv